In [0]:
import yaml

CONFIG_PATH = "/Workspace/Users/avranilset@gmail.com/capstone-project/config/config.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Extract config
CATALOG = config["databricks"]["catalog"]

BRONZE_DB = config["databricks"]["schemas"]["bronze"]
SILVER_DB = config["databricks"]["schemas"]["silver"]
GOLD_DB = config["databricks"]["schemas"]["gold"]
LOGS_DB = config["databricks"]["schemas"]["logs"]

NULL_THRESHOLD_PCT = config["data_quality"]["null_threshold_pct"]
DUPLICATE_THRESHOLD_PCT = config["data_quality"]["duplicate_threshold_pct"]
RECONCILIATION_TOLERANCE_PCT = config["data_quality"]["reconciliation_tolerance_pct"]

LOG_TABLE = config["data_quality"]["log_table"]

# Gold tables (important)
FACT_SALES = config["gold"]["fact_sales"]
DIM_CUSTOMER = config["gold"]["dim_customer"]
DIM_PRODUCT = config["gold"]["dim_product"]

print("Config loaded successfully")

Config loaded successfully


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan
import json
from datetime import datetime

In [0]:
dbutils.widgets.text("layer", "gold")
layer = dbutils.widgets.get("layer")  # "silver" or "gold"


print(f"DATA QUALITY — {layer.upper()} LAYER")


results = []

DATA QUALITY — GOLD LAYER


In [0]:
if layer == "silver":

    print("\n--- Reconciliation Checks ---")

    reconciliation_pairs = [
        (
            config["datasets"]["customers"]["bronze_table"],
            config["datasets"]["customers"]["silver_table"]
        ),
        (
            config["datasets"]["products"]["bronze_table"],
            config["datasets"]["products"]["silver_table"]
        ),
        (
            config["datasets"]["orders"]["bronze_table"],
            config["datasets"]["orders"]["silver_table"]
        ),
        (
            config["datasets"]["order_items"]["bronze_table"],
            config["datasets"]["order_items"]["silver_table"]
        ),
    ]

    for bt, st in reconciliation_pairs:
        bronze_table = f"{CATALOG}.{BRONZE_DB}.{bt}"
        silver_table = f"{CATALOG}.{SILVER_DB}.{st}"

        bc = spark.table(bronze_table).count()
        sc = spark.table(silver_table).count()

        pct = ((bc - sc) / bc * 100) if bc > 0 else 0
        status = "PASS" if pct <= RECONCILIATION_TOLERANCE_PCT else "WARN"

        print(f"  {'✓' if status=='PASS' else '⚠'} {bronze_table} → {silver_table} | {bc:,} → {sc:,} | Diff: {pct:.2f}%")

        results.append({
            "check_type": "RECONCILIATION",
            "table": f"{bronze_table} → {silver_table}",
            "column": "count",
            "status": status,
            "details": {"bronze": bc, "silver": sc, "diff_pct": round(pct, 2)},
            "timestamp": datetime.now().isoformat()
        })


elif layer == "gold":

    print("\n--- Uniqueness Checks ---")

    uniqueness_checks = [
    (config["gold"]["fact_sales"], "sale_id"),
    (config["gold"]["dim_customer"], "surrogate_key"),
    (config["gold"]["dim_product"], "surrogate_key"),
    ]

    for table, pk in uniqueness_checks:
        full_table = f"{CATALOG}.{GOLD_DB}.{table}"

        df = spark.table(full_table)
        total = df.count()
        distinct = df.select(pk).distinct().count()

        dups = total - distinct
        pct = (dups / total * 100) if total > 0 else 0
        status = "PASS" if pct <= DUPLICATE_THRESHOLD_PCT else "FAIL"

        print(f"  {'✓' if status=='PASS' else '✗'} {full_table}.{pk} | Dups: {dups} ({pct:.2f}%)")

        results.append({
            "check_type": "UNIQUENESS",
            "table": full_table,
            "column": pk,
            "status": status,
            "details": {"total": total, "distinct": distinct, "dups": dups},
            "timestamp": datetime.now().isoformat()
        })


    print("\n--- Completeness Checks ---")

    completeness_checks = [
    (config["gold"]["fact_sales"], ["order_id","customer_sk","product_sk","order_date","total_amount","quantity","unit_price"]),
    (config["gold"]["dim_customer"], ["customer_id","full_name","email"]),
    (config["gold"]["dim_product"], ["product_id","product_name","category"]),
    ]  

    for table, cols in completeness_checks:
        full_table = f"{CATALOG}.{GOLD_DB}.{table}"
        df = spark.table(full_table)
        total = df.count()

        for c in cols:
            if c == "order_date":
                nulls = df.filter(
                    col(c).isNull() | (col(c).cast("string") == "")
                ).count()
            else:
                nulls = df.filter(
                    col(c).isNull() |
                    (col(c).cast("string") == "") |
                    isnan(col(c).try_cast("double"))
                ).count()

            pct = (nulls / total * 100) if total > 0 else 0
            status = "PASS" if pct <= NULL_THRESHOLD_PCT else "FAIL"

            print(f"  {'✓' if status=='PASS' else '✗'} {full_table}.{c} | NULLs: {nulls} ({pct:.2f}%)")

            results.append({
                "check_type": "COMPLETENESS",
                "table": full_table,
                "column": c,
                "status": status,
                "details": {"nulls": nulls, "pct": round(pct,2)},
                "timestamp": datetime.now().isoformat()
            })


    print("\n--- Business Rule Checks ---")

    fact_table = config["gold"]["fact_sales"]
    dim_customer_table = config["gold"]["dim_customer"]
    dim_product_table = config["gold"]["dim_product"]

    fact = spark.table(f"{CATALOG}.{GOLD_DB}.{fact_table}")
    dim_c = spark.table(f"{CATALOG}.{GOLD_DB}.{dim_customer_table}")
    dim_p = spark.table(f"{CATALOG}.{GOLD_DB}.{dim_product_table}")

    bad = fact.filter(col("total_amount") <= 0).count()
    status = "PASS" if bad == 0 else "FAIL"
    print(f"  {'Pass' if status=='PASS' else 'Fail'} total_amount > 0 | {bad}")

    results.append({
        "check_type": "BUSINESS_RULE",
        "table": f"{CATALOG}.{GOLD_DB}.{fact_table}",
        "column": "total_amount",
        "status": status,
        "details": {"invalid": bad},
        "timestamp": datetime.now().isoformat()
    })

    orphans = fact.select("customer_sk").distinct().subtract(
    dim_c.filter(col("is_current") == True)
        .select(col("surrogate_key").alias("customer_sk"))
        .distinct()
        ).count()

    status = "PASS" if orphans == 0 else "FAIL"
    print(f"  {'Pass' if status=='PASS' else 'Fail'} FK customer_id | {orphans}")

    results.append({
        "check_type": "BUSINESS_RULE",
        "table": f"{CATALOG}.{GOLD_DB}.{fact_table}",
        "column": "customer_id",
        "status": status,
        "details": {"orphans": orphans},
        "timestamp": datetime.now().isoformat()
    })

    orphans = fact.select("product_sk").distinct().subtract(
    dim_p.select(col("surrogate_key").alias("product_sk")).distinct()
).count()

    status = "PASS" if orphans == 0 else "FAIL"
    print(f"  {'PASS' if status=='PASS' else 'FAIL'} FK product_id | {orphans}")

    results.append({
        "check_type": "BUSINESS_RULE",
        "table": f"{CATALOG}.{GOLD_DB}.{fact_table}",
        "column": "product_id",
        "status": status,
        "details": {"orphans": orphans},
        "timestamp": datetime.now().isoformat()
    })


--- Uniqueness Checks ---
  ✓ capstone_project_catalog.gold.fact_sales.sale_id | Dups: 0 (0.00%)
  ✓ capstone_project_catalog.gold.dim_customer.surrogate_key | Dups: 0 (0.00%)
  ✓ capstone_project_catalog.gold.dim_product.surrogate_key | Dups: 0 (0.00%)

--- Completeness Checks ---
  ✓ capstone_project_catalog.gold.fact_sales.order_id | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.fact_sales.customer_sk | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.fact_sales.product_sk | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.fact_sales.order_date | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.fact_sales.total_amount | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.fact_sales.quantity | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.fact_sales.unit_price | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.dim_customer.customer_id | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.dim_customer.full_name | NULLs: 0 (0.00%)
  ✓ capstone_project_catalog.gold.dim_

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("check_type", StringType(), True),
    StructField("table", StringType(), True),
    StructField("column", StringType(), True),
    StructField("status", StringType(), True),
    StructField("details", StringType(), True),
    StructField("timestamp", StringType(), True)
])


In [0]:
print("\n--- Saving DQ Logs ---")

log_table = config["data_quality"]["log_table"]

log_df = spark.createDataFrame(results, schema=schema)

log_df.write.format("delta") \
    .mode("append") \
    .saveAsTable(f"{CATALOG}.{LOGS_DB}.{log_table}")

failures = [r for r in results if r.get("status") == "FAIL"]

print(f"DQ RESULT: {'PASS' if len(failures)==0 else 'FAIL'} | {len(results)} checks | {len(failures)} failures")


--- Saving DQ Logs ---
DQ RESULT: PASS | 19 checks | 0 failures
